In [ ]:
import os, re, warnings
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, confusion_matrix

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

BASE      = Path('./Biometrics')  # the module's data folder, with 'training' and 'test' inside
TRAIN_DIR = BASE / 'training'
TEST_DIR  = BASE / 'test'
print('Libraries loaded.')
print(f'Training: {TRAIN_DIR}')
print(f'Test:     {TEST_DIR}')


# Parse training data

In [ ]:
pattern = re.compile(r'(\d+)z(\d+)p([fs])\.jpg', re.IGNORECASE)
records = []
for fname in sorted(TRAIN_DIR.iterdir()):
    m = pattern.match(fname.name)
    if m:
        records.append({'path':str(fname),'filename':fname.name,
                        'subject':m.group(1),'seq':m.group(2),'view':m.group(3).lower()})
df_train = pd.DataFrame(records)
print(f'Training images: {len(df_train)}')
print(df_train.groupby(['subject','view']).size().unstack(fill_value=0))
print('Sequences/subject:')
print(df_train.groupby('subject')['seq'].nunique())


# List test images

In [ ]:
test_paths = sorted(TEST_DIR.iterdir())
print(f'Test images: {len(test_paths)}')
for p in test_paths:
    print(' ', p.name)


# Silhouette extraction

In [ ]:
def extract_silhouette(img_path):
    img = cv2.imread(str(img_path))
    if img is None: return None, None
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    green = cv2.inRange(hsv, np.array([35,40,40]), np.array([85,255,255]))
    pmask = cv2.bitwise_not(green)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(7,7))
    pmask = cv2.morphologyEx(pmask, cv2.MORPH_CLOSE, k)
    pmask = cv2.morphologyEx(pmask, cv2.MORPH_OPEN,  k)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(pmask)
    if n <= 1: return pmask, None
    lg = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    clean = ((labels==lg)*255).astype(np.uint8)
    x,y,w,h = (stats[lg,cv2.CC_STAT_LEFT], stats[lg,cv2.CC_STAT_TOP],
               stats[lg,cv2.CC_STAT_WIDTH], stats[lg,cv2.CC_STAT_HEIGHT])
    return clean, (y,y+h,x,x+w)

def crop_sil(mask, bbox):
    y1,y2,x1,x2 = bbox
    return mask[y1:y2,x1:x2]

print('Extractor ready.')


# Visualise silhouettes

In [ ]:
fig, axes = plt.subplots(3,4,figsize=(14,10))
for ri, subj in enumerate(['016','018','021']):
    for ci, (_,r) in enumerate(df_train[df_train['subject']==subj].head(2).iterrows()):
        mask, bbox = extract_silhouette(r['path'])
        img = cv2.cvtColor(cv2.imread(r['path']),cv2.COLOR_BGR2RGB)
        axes[ri,ci*2].imshow(img)
        axes[ri,ci*2].set_title("Subj "+subj+" - "+('Front' if r['view']=='f' else 'Side'))
        axes[ri,ci*2].axis('off')
        axes[ri,ci*2+1].imshow(crop_sil(mask,bbox),cmap='gray')
        axes[ri,ci*2+1].set_title('Silhouette')
        axes[ri,ci*2+1].axis('off')
plt.suptitle('Extracted Silhouettes (HSV Green-Screen Masking)',fontsize=13)
plt.tight_layout()
plt.savefig('silhouettes.png',bbox_inches='tight',dpi=120)
plt.show()


# Feature extraction functions

In [ ]:
SIL_H, SIL_W = 256, 64

def norm_sil(cropped):
    r = cv2.resize(cropped,(SIL_W,SIL_H),interpolation=cv2.INTER_AREA)
    return (r>127).astype(np.float32)

def geom_features(ns, bbox):
    H,W = ns.shape
    wp = ns.sum(axis=1)/W
    fracs = [0.05,0.15,0.30,0.45,0.55,0.75,0.92]
    samp = [float(wp[max(0,int(f*H)-3):int(f*H)+4].mean()) for f in fracs]
    y1,y2,x1,x2 = bbox
    aspect = (y2-y1)/(x2-x1+1e-6)
    area = float(ns.sum()); fill = area/(H*W)
    si = ns.astype(np.int32)
    per = float(np.abs(np.diff(si,axis=1)).sum()+np.abs(np.diff(si,axis=0)).sum())
    comp = 4*np.pi*area/(per**2+1e-6)
    Mv = cv2.moments(ns); m0 = Mv['m00']+1e-6
    cx=Mv['m10']/m0/W; cy=Mv['m01']/m0/H
    mu20=Mv['mu20']/m0/W**2; mu02=Mv['mu02']/m0/H**2; mu11=Mv['mu11']/m0/(W*H)
    shd = float(wp[:int(0.25*H)].max())
    wsr = samp[3]/(samp[1]+1e-6)
    sc = np.array([aspect,fill,comp,cx,cy,mu20,mu02,mu11,shd,wsr]+samp,dtype=np.float32)
    return wp.astype(np.float32), sc

def feat_pipeline(path):
    mask,bbox = extract_silhouette(path)
    if bbox is None: return None,None,None
    ns = norm_sil(crop_sil(mask,bbox))
    wp,sc = geom_features(ns,bbox)
    return ns, wp, sc

print('Feature functions ready.')


# Extract all training features

In [ ]:
print('Extracting training features...')
sils,geoms,vidx = [],[],[]
for i,(_,row) in enumerate(df_train.iterrows()):
    ns,wp,sc = feat_pipeline(row['path'])
    if ns is not None:
        sils.append(ns.flatten())
        geoms.append(np.concatenate([wp,sc]))
        vidx.append(i)
df_v = df_train.iloc[vidx].reset_index(drop=True)
X_sil  = np.array(sils, dtype=np.float32)
X_geom = np.array(geoms,dtype=np.float32)
print(f'Valid: {len(df_v)} | Sil: {X_sil.shape} | Geom: {X_geom.shape}')


# Width profile visualisation

In [ ]:
fig,axes = plt.subplots(1,2,figsize=(14,6))
subj='021'
rows_s = df_v[(df_v['subject']==subj)&(df_v['view']=='f')]
cols = plt.cm.Blues(np.linspace(0.4,1.0,max(len(rows_s),2)))
for ci,(_,r) in enumerate(rows_s.iterrows()):
    _,wp,_ = feat_pipeline(r['path'])
    if wp is not None:
        axes[0].plot(wp,range(len(wp)),color=cols[ci],lw=1.5,label='seq '+r['seq'])
axes[0].set_title('Intra-class -- Subj '+subj+' (Front)',fontsize=12)
axes[0].set_xlabel('Norm Width'); axes[0].set_ylabel('Height (0=top)')
axes[0].invert_yaxis(); axes[0].legend(fontsize=8); axes[0].grid(True,alpha=0.3)
cmap=plt.cm.tab10; subs=sorted(df_v['subject'].unique())
for ci,sid in enumerate(subs):
    r=df_v[(df_v['subject']==sid)&(df_v['view']=='f')].iloc[0]
    _,wp,_=feat_pipeline(r['path'])
    if wp is not None:
        axes[1].plot(wp,range(len(wp)),color=cmap(ci/len(subs)),lw=1.5,label='S'+sid)
axes[1].set_title('Inter-class -- All Subjects (Front)',fontsize=12)
axes[1].set_xlabel('Norm Width'); axes[1].set_ylabel('Height (0=top)')
axes[1].invert_yaxis(); axes[1].legend(fontsize=8); axes[1].grid(True,alpha=0.3)
plt.suptitle('Width Profiles: Intra-class vs Inter-class Variation',fontsize=13)
plt.tight_layout()
plt.savefig('width_profiles.png',bbox_inches='tight',dpi=120)
plt.show()


# PCA / Eigensilhouettes

In [ ]:
sc_geom = StandardScaler()
Xg = sc_geom.fit_transform(X_geom)
N_COMP = min(len(df_v)-1,40)
pca = PCA(n_components=N_COMP,whiten=True,random_state=42)
Xp = pca.fit_transform(X_sil)
cv = np.cumsum(pca.explained_variance_ratio_)
k95 = int(np.searchsorted(cv,0.95))+1
print(f'Components: {N_COMP}  95%@k={k95}  total var={cv[-1]*100:.1f}%')


# PCA variance plots

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4))
axes[0].bar(range(1,N_COMP+1),pca.explained_variance_ratio_*100,color='steelblue',alpha=0.7)
axes[0].set(xlabel='PC',ylabel='Var %',title='Scree Plot'); axes[0].grid(True,alpha=0.3)
axes[1].plot(range(1,N_COMP+1),cv*100,'b-o',ms=4)
axes[1].axhline(95,color='r',ls='--',label='95%'); axes[1].axvline(k95,color='r',ls='--')
axes[1].set(xlabel='k',ylabel='Cum Var %',title='Cumulative Variance')
axes[1].legend(); axes[1].grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig('pca_variance.png',bbox_inches='tight',dpi=120)
plt.show()


# Eigensilhouette visualisation

In [ ]:
n_show=min(9,N_COMP)
fig,axes=plt.subplots(2,5,figsize=(14,6))
axes[0,0].imshow(pca.mean_.reshape(SIL_H,SIL_W),cmap='gray',vmin=0,vmax=1)
axes[0,0].set_title('Mean sil',fontsize=9); axes[0,0].axis('off')
for i in range(n_show):
    ax=axes[(i+1)//5,(i+1)%5]
    ax.imshow(pca.components_[i].reshape(SIL_H,SIL_W),cmap='RdBu_r')
    ax.set_title(f'PC{i+1} ({pca.explained_variance_ratio_[i]*100:.1f}%)',fontsize=9)
    ax.axis('off')
plt.suptitle('Mean Silhouette and Top Eigensilhouettes (PCA)',fontsize=12)
plt.tight_layout()
plt.savefig('eigensilhouettes.png',bbox_inches='tight',dpi=120)
plt.show()


# Combine features

In [ ]:
sc_pca = StandardScaler()
Xps = sc_pca.fit_transform(Xp)
X_comb = np.hstack([Xps, Xg])
print(f'Combined dim: {X_comb.shape[1]}  (PCA:{Xps.shape[1]}  Geom:{Xg.shape[1]})')


# LOSO cross-validation

In [ ]:
def nn_cls(probe, gal_f, gal_l):
    dists = np.linalg.norm(gal_f-probe,axis=1)
    seen,ranked = set(),[]
    for idx in np.argsort(dists):
        l=gal_l[idx]
        if l not in seen: ranked.append(l); seen.add(l)
    return ranked, dists

def loso_cv(Xf, df):
    ns_val = df['subject'].nunique()
    yt,yp,gen,imp,rc = [],[],[],[],defaultdict(int)
    np_ = 0
    for sid in df['seq'].unique():
        pm=df['seq']==sid; gm=~pm
        Xpr=Xf[pm.values]; Xga=Xf[gm.values]
        lp=df.loc[pm,'subject'].values; lg=df.loc[gm,'subject'].values
        if len(Xga)==0: continue
        for feat,tl in zip(Xpr,lp):
            ranked,dists=nn_cls(feat,Xga,lg)
            yt.append(tl); yp.append(ranked[0]); np_+=1
            for rk in range(1,ns_val+1):
                if tl in ranked[:rk]: rc[rk]+=1
            for gl,d in zip(lg,dists):
                (gen if gl==tl else imp).append(-d)
    ccr=sum(a==b for a,b in zip(yt,yp))/max(np_,1)
    cmc={r:rc[r]/max(np_,1) for r in range(1,ns_val+1)}
    return {'yt':yt,'yp':yp,'ccr':ccr,'gen':gen,'imp':imp,'cmc':cmc,'n':np_}

print('Running LOSO-CV...')
res = loso_cv(X_comb, df_v)
n_subj = df_v['subject'].nunique()
print(f'Probes: {res["n"]}  CCR: {res["ccr"]*100:.1f}%  Baseline: {100/n_subj:.1f}%')


# Per-subject CCR

In [ ]:
subs_u = sorted(df_v['subject'].unique())
ya = np.array(res['yt']); yp = np.array(res['yp'])
rows=[]
for s in subs_u:
    m=ya==s; tot=int(m.sum()); cor=int((yp[m]==s).sum()) if tot else 0
    rows.append({'Subject':s,'Probes':tot,'Correct':cor,'CCR%':f'{cor/max(tot,1)*100:.1f}'})
print(pd.DataFrame(rows).set_index('Subject').to_string())


# Score distribution histograms

In [ ]:
gen=np.array(res['gen']); imp=np.array(res['imp'])
fig,ax=plt.subplots(figsize=(10,5))
ax.hist(gen,bins=60,alpha=0.65,color='#2ecc71',density=True,label=f'Genuine (n={len(gen)})')
ax.hist(imp,bins=60,alpha=0.65,color='#e74c3c',density=True,label=f'Impostor (n={len(imp)})')
ax.set(xlabel='Score (-Euclidean Distance)',ylabel='Density',
       title='Intra-class vs Inter-class Score Distributions')
ax.legend(fontsize=11); ax.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig('score_distributions.png',bbox_inches='tight',dpi=120)
plt.show()
print(f'Genuine  mean={gen.mean():.3f} std={gen.std():.3f}')
print(f'Impostor mean={imp.mean():.3f} std={imp.std():.3f}')


# ROC curve and EER

In [ ]:
ys=np.concatenate([gen,imp]); yl=np.concatenate([np.ones(len(gen)),np.zeros(len(imp))])
fpr,tpr,thr = roc_curve(yl,ys); ra=auc(fpr,tpr)
fnr=1-tpr; ei=int(np.nanargmin(np.abs(fpr-fnr)))
eer=float((fpr[ei]+fnr[ei])/2); ethr=float(thr[ei])
print(f'AUC={ra:.4f}  EER={eer*100:.2f}%  thr={ethr:.4f}')

fig,ax=plt.subplots(figsize=(8,7))
ax.plot(fpr,tpr,color='steelblue',lw=2,label=f'ROC (AUC={ra:.3f})')
ax.plot([0,1],[0,1],'k--',lw=1,label='Random')
ax.plot(fpr[ei],tpr[ei],'ro',ms=10,label=f'EER={eer*100:.2f}%',zorder=5)
ax.annotate(f'EER={eer*100:.2f}%',xy=(fpr[ei],tpr[ei]),
            xytext=(fpr[ei]+0.08,tpr[ei]-0.10),fontsize=11,
            arrowprops=dict(arrowstyle='->',color='red'))
ax.set(xlabel='False Acceptance Rate',ylabel='True Acceptance Rate',title='ROC Curve')
ax.legend(fontsize=11); ax.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png',bbox_inches='tight',dpi=120)
plt.show()


# CCR at EER

In [ ]:
ccr_eer = float((gen>ethr).mean())
print(f'CCR at EER (thr={ethr:.4f}): {ccr_eer*100:.2f}%')


# CMC curve

In [ ]:
ranks=list(range(1,n_subj+1))
cmcv=[res['cmc'].get(r,0)*100 for r in ranks]
fig,ax=plt.subplots(figsize=(8,6))
ax.plot(ranks,cmcv,'bo-',lw=2,ms=8); ax.fill_between(ranks,cmcv,alpha=0.15,color='blue')
ax.axhline(100/n_subj,color='red',ls='--',label=f'Random ({100/n_subj:.1f}%)')
ax.set(xlabel='Rank',ylabel='CCR (%)',title='CMC Curve')
ax.set_xticks(ranks); ax.set_ylim([0,105]); ax.legend(); ax.grid(True,alpha=0.3)
for r,v in zip(ranks,cmcv):
    ax.annotate(f'{v:.0f}%',(r,v),textcoords='offset points',xytext=(0,8),ha='center',fontsize=9)
plt.tight_layout()
plt.savefig('cmc_curve.png',bbox_inches='tight',dpi=120)
plt.show()
print('CMC:')
for r,v in zip(ranks,cmcv): print(f'  Rank-{r}: {v:.1f}%')


# Confusion matrix

In [ ]:
cm=confusion_matrix(ya,yp,labels=subs_u)
cmn=cm.astype(float)/(cm.sum(axis=1,keepdims=True)+1e-6)
fig,ax=plt.subplots(figsize=(8,7))
sns.heatmap(cmn,annot=True,fmt='.2f',cmap='Blues',
            xticklabels=subs_u,yticklabels=subs_u,vmin=0,vmax=1,ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix\nCCR={:.1f}%'.format(res['ccr']*100))
plt.tight_layout()
plt.savefig('confusion_matrix.png',bbox_inches='tight',dpi=120)
plt.show()


# Summary

In [ ]:
print('='*52)
print('BIOMETRIC SYSTEM PERFORMANCE SUMMARY')
print('='*52)
print(f'Dataset:    Southampton Gait DB - {n_subj} subjects, {len(df_v)} images')
print(f'Features:   PCA eigensilhouettes + geometric body measurements')
print(f'Classifier: 1-Nearest Neighbour (Euclidean distance)')
print(f'Protocol:   Leave-one-sequence-out cross-validation')
print()
print(f'CCR Rank-1: {res["ccr"]*100:.1f}%  Rank-2: {res["cmc"].get(2,0)*100:.1f}%  Rank-3: {res["cmc"].get(3,0)*100:.1f}%')
print(f'Random baseline: {100/n_subj:.1f}%')
print(f'EER: {eer*100:.2f}%  CCR@EER: {ccr_eer*100:.2f}%  AUC: {ra:.4f}')
print('='*52)


# Test set predictions

In [ ]:
print('Classifying test images...')
tres=[]
for p in sorted(TEST_DIR.iterdir()):
    ns2,wp,sc = feat_pipeline(str(p))
    if ns2 is None:
        tres.append({'file':p.name,'pred':'FAIL','top3':[],'d':None}); continue
    pf = sc_pca.transform(pca.transform(ns2.flatten().reshape(1,-1)))
    gf = sc_geom.transform(np.concatenate([wp,sc]).reshape(1,-1))
    cb = np.hstack([pf,gf])
    ranked,dists = nn_cls(cb[0],X_comb,df_v['subject'].values)
    bd = float(dists[df_v['subject'].values==ranked[0]].min())
    tres.append({'file':p.name,'pred':ranked[0],'top3':ranked[:3],'d':round(bd,3)})
df_test=pd.DataFrame(tres)
print(df_test[['file','pred','top3','d']].to_string())


# Visualise test predictions

In [ ]:
nsh=min(len(tres),12); nc=4; nr=(nsh+nc-1)//nc
fig,axes=plt.subplots(nr,nc,figsize=(14,nr*3.5))
axes=axes.flatten()
for i,r in enumerate(tres[:nsh]):
    img=cv2.cvtColor(cv2.imread(str(TEST_DIR/r['file'])),cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)
    t3=', '.join(r['top3'][:3]) if r['top3'] else 'N/A'
    axes[i].set_title(r['file']+'\nPred: '+r['pred']+'\nTop3: '+t3,fontsize=8)
    axes[i].axis('off')
for j in range(nsh,len(axes)): axes[j].axis('off')
plt.suptitle('Test Set Predictions',fontsize=13)
plt.tight_layout()
plt.savefig('test_predictions.png',bbox_inches='tight',dpi=100)
plt.show()
